In [6]:
import pandas as pd

# Load the CSV
df = pd.read_csv("scam-dialogue_train.csv")  # path to your CSV

# Check data
print(df.head())

# Split features and labels
X = df['dialogue']      # text
y = df['label']         # 1 = scam, 0 = non-scam

                                            dialogue type  label
0  caller: Hello, is this John? receiver: Yes, it...  ssn      1
1  caller: Hello, is this John? receiver: Yeah, t...  ssn      1
2  caller: Hello, is this Mr. Johnson? receiver: ...  ssn      1
3  caller: Hello, is this John? receiver: Yeah, t...  ssn      1
4  caller: Hello, this is Officer Johnson from th...  ssn      1


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize text
train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True, max_length=128)

In [9]:
import torch

class ScamDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

train_dataset = ScamDataset(train_encodings, y_train)
test_dataset = ScamDataset(test_encodings, y_test)

In [10]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
optimizer = AdamW(model.parameters(), lr=5e-5)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

model.train()
for epoch in range(3):  # 3 epochs for testing
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} finished")

Epoch 1 finished
Epoch 2 finished
Epoch 3 finished


In [12]:
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

model.eval()
test_loader = DataLoader(test_dataset, batch_size=16)
preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        pred_labels = torch.argmax(outputs.logits, dim=1)
        preds.extend(pred_labels.cpu().numpy())

acc = accuracy_score(y_test, preds)
print("Test Accuracy:", acc)

Test Accuracy: 1.0


In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def predict_scam(text):
    model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)

    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()

    return prediction